# Patient-safe splitting — Subset Data

## Summary

- All **6,864 images from 422 patients** are assigned to five reproducible cross-validation folds.
- Each patient belongs to exactly one fold, preventing patient leakage between training and validation.
- The folds contain 1,370–1,381 images and 25–26 malignant images each.
- The labelled subset test set remains outside cross-validation; only its patient identifiers are read to confirm zero train–test patient overlap.
- The fold-labelled training data is saved as `data/subset/processed/train_folds.csv`.


## Context & Methods

Used five-fold `StratifiedGroupKFold`: the image-level `target` supplies the stratification labels and `patient_id` supplies the non-overlapping groups.

### Key assumptions

- `target = 1` denotes a malignant image.
- `patient_id` is the required leakage-prevention unit.
- Five folds, shuffling, and random seed 42 are fixed before modelling.
- Fold balance is approximate because patients contribute different numbers and classes of images. Patient isolation takes priority over identical fold sizes.
- The labelled subset test data is reserved for one final evaluation and is not used to create or select folds.


## Data

### 1. Configure reproducible project paths


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SOURCE_DIR = PROJECT_ROOT / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from melanoma_detection.splitting import assign_patient_folds

PROCESSED_DIR = PROJECT_ROOT / "data" / "subset" / "processed"
TRAIN_CLEAN_PATH = PROCESSED_DIR / "train_clean.csv"
TEST_CLEAN_PATH = PROCESSED_DIR / "test_clean.csv"
TRAIN_FOLDS_PATH = PROCESSED_DIR / "train_folds.csv"

N_SPLITS = 5
RANDOM_STATE = 42

print(f"Project root: {PROJECT_ROOT}")
print(f"Training input: {TRAIN_CLEAN_PATH}")
print(f"Fold output: {TRAIN_FOLDS_PATH}")


Project root: C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma
Training input: C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\subset\processed\train_clean.csv
Fold output: C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\subset\processed\train_folds.csv


### 2. Load the cleaned subset training data


In [2]:
train = pd.read_csv(TRAIN_CLEAN_PATH, dtype={"age_approx": "string"})

patient_targets = train.groupby("patient_id")["target"].max()
input_summary = pd.Series(
    {
        "images": len(train),
        "patients": train["patient_id"].nunique(),
        "malignant images": int(train["target"].sum()),
        "positive patients": int(patient_targets.sum()),
    },
    name="count",
)
print(input_summary.to_string())


images               6864
patients              422
malignant images      128
positive patients      88


## Results

### 3. Assign five stratified patient-grouped folds


In [3]:
train_folds = assign_patient_folds(
    train,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    target_col="target",
    group_col="patient_id",
)

print(f"Assigned folds: {sorted(train_folds['fold'].unique().tolist())}")
print(f"Fold dtype: {train_folds['fold'].dtype}")


Assigned folds: [0, 1, 2, 3, 4]
Fold dtype: int8


### 4. Audit fold balance at image and patient levels


In [4]:
image_summary = train_folds.groupby("fold")["target"].agg(
    images="size",
    malignant_images="sum",
    image_malignancy_rate="mean",
)

patient_fold_targets = train_folds.groupby(["fold", "patient_id"])["target"].max()
patient_summary = patient_fold_targets.groupby("fold").agg(
    patients="size",
    positive_patients="sum",
    patient_positivity_rate="mean",
)

fold_summary = image_summary.join(patient_summary)
display_summary = fold_summary.copy()
display_summary["image_malignancy_rate"] = display_summary["image_malignancy_rate"].map(
    lambda value: f"{value:.2%}"
)
display_summary["patient_positivity_rate"] = display_summary["patient_positivity_rate"].map(
    lambda value: f"{value:.2%}"
)
display_summary


,images,malignant_images,image_malignancy_rate,patients,positive_patients,patient_positivity_rate
fold,,,,,,
0,1370,25,1.82%,84,17,20.24%
1,1372,26,1.90%,84,15,17.86%
2,1381,26,1.88%,87,21,24.14%
3,1371,25,1.82%,82,18,21.95%
4,1370,26,1.90%,85,17,20.00%


### 5. Prove patient isolation and complete row assignment


In [5]:
patient_fold_counts = train_folds.groupby("patient_id")["fold"].nunique()
patients_in_multiple_folds = int(patient_fold_counts.gt(1).sum())
unassigned_rows = int(train_folds["fold"].isna().sum())

assert len(train_folds) == len(train), "Row count changed during fold assignment."
assert train_folds["image_name"].equals(train["image_name"]), (
    "Image order changed during fold assignment."
)
assert set(train_folds["fold"]) == set(range(N_SPLITS)), "Unexpected fold labels."
assert unassigned_rows == 0, "At least one row has no fold."
assert patients_in_multiple_folds == 0, "Patient leakage exists between folds."
assert all(set(group["target"]) == {0, 1} for _, group in train_folds.groupby("fold")), (
    "At least one fold is missing a target class."
)

print(f"Unassigned rows: {unassigned_rows}")
print(f"Patients appearing in multiple folds: {patients_in_multiple_folds}")


Unassigned rows: 0
Patients appearing in multiple folds: 0


### 6. Confirm the reserved test patients are disjoint

Only `patient_id` is loaded from the test file. Test targets and other test fields do not influence fold creation or validation.


In [6]:
test_patient_ids = pd.read_csv(TEST_CLEAN_PATH, usecols=["patient_id"])["patient_id"]
train_patient_set = set(train_folds["patient_id"])
test_patient_set = set(test_patient_ids)
overlapping_patients = train_patient_set & test_patient_set

assert not overlapping_patients, "Patients overlap between subset train and test."
print(f"Training patients: {len(train_patient_set):,}")
print(f"Reserved test patients: {len(test_patient_set):,}")
print(f"Train–test patient overlap: {len(overlapping_patients)}")


Training patients: 422
Reserved test patients: 106
Train–test patient overlap: 0


### 7. Save and reconcile the fold-labelled training data


In [7]:
train_folds.to_csv(TRAIN_FOLDS_PATH, index=False)
saved_folds = pd.read_csv(TRAIN_FOLDS_PATH, dtype={"age_approx": "string"})

assert len(saved_folds) == len(train), "Saved row count differs from the input."
assert list(saved_folds.columns) == [*train.columns, "fold"], (
    "Saved columns differ from the expected schema."
)
assert saved_folds["image_name"].equals(train["image_name"]), (
    "Saved image order differs from the input."
)
assert saved_folds["target"].equals(train["target"]), (
    "Saved targets differ from the input."
)

print(f"Wrote: {TRAIN_FOLDS_PATH}")
print(f"Saved shape: {saved_folds.shape[0]:,} rows x {saved_folds.shape[1]} columns")


Wrote: C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\subset\processed\train_folds.csv
Saved shape: 6,864 rows x 9 columns


## Takeaways

1. Five patient-safe folds were created without changing row order, image identifiers, targets, or source columns.
2. Image-level stratification produced tightly balanced fold sizes and 25–26 malignant images per fold.
3. Positive-patient counts vary from 15 to 21 because patients contribute different numbers and classes of images; this is reported transparently rather than corrected using test information.
4. No patient appears in more than one cross-validation fold or in both the training and reserved test datasets.
5. All model selection must use these training folds. The labelled subset test data remains reserved for the final evaluation notebook.
